# DSPy Basics: Programming, Not Prompting [Agent Patterns - Module 09]

> **MLCourse - Agentic AI - Agent Patterns**

Everything you have written so far in this course put *English strings*
at the centre of the program. You wrote a prompt, ran it, squinted at the
output, edited a word, ran it again. That loop works, but it has three
problems:

1. It does not scale. Ten prompts across an agent = ten hand-tuned strings.
2. It is not measurable. "That reads better" is not a number.
3. It does not survive a model swap. Change the model and your careful
   phrasing is tuned for a model you are no longer using.

**DSPy** takes the opposite position: you declare *what* goes in and *what*
comes out, and the framework generates and improves the actual prompt text
for you - guided by a metric and a handful of examples.

### What you will learn

1. What a DSPy **Signature** is and why it replaces a prompt string.
2. How to point DSPy at **Groq** through LiteLLM.
3. `dspy.Predict` vs `dspy.ChainOfThought`.
4. How to see the *real* prompt DSPy sent to the model.
5. Where DSPy fits next to LangChain / LangGraph / CrewAI.

### Key takeaways

- A Signature is a typed interface: `question -> answer`.
- The prompt is a compiled artifact, not source code you hand-edit.
- `inspect_history()` is your window into what was actually sent.

### Setup: imports, environment, track discovery


In [ ]:
import os
import time
import random
from pathlib import Path
from dotenv import load_dotenv

def _find_track(depth=6):
    p = Path.cwd()
    for _ in range(depth):
        if (p / "03_agentic_ai").is_dir():
            return p
        p = p.parent
    return Path.cwd()

TRACK = _find_track()
# The .env lives INSIDE the track folder, not the repo root that
# _find_track() returns - joining ".env" onto TRACK alone is a silent no-op.
load_dotenv(TRACK / "03_agentic_ai" / ".env", override=False)

GROQ_API_KEY = os.environ["GROQ_API_KEY"]   # loud failure if missing, by design
MODEL = "qwen/qwen3.8-27b"                   # Groq-hosted; never OpenAI

print(f"Track root : {TRACK}")
print(f"Model      : {MODEL} (via Groq)")
print(f"Key loaded : {bool(GROQ_API_KEY)}")


### Point DSPy at Groq (through LiteLLM)


In [ ]:
# dspy.LM is a thin wrapper over LiteLLM. The "groq/" prefix is the LiteLLM
# provider route - the rest is the Groq model id.

import dspy

lm = dspy.LM(
    f"groq/{MODEL}",
    api_key=GROQ_API_KEY,
    temperature=0.0,      # deterministic-ish: we are going to MEASURE things
    max_tokens=700,
    num_retries=5,        # LiteLLM backs off on 429 (Groq free tier = 8000 TPM)
)
dspy.configure(lm=lm)

print("DSPy configured.")
print("dspy version:", dspy.__version__)


### 1. The Signature

A **Signature** declares the shape of one LLM call: named inputs, named
outputs, and (optionally) a short description of each. The shorthand form
is a string:

```
"question -> answer"
"document -> summary"
"message -> category, confidence"
```

Notice what is *missing*: no "You are a helpful assistant", no "think step
by step", no "respond only with JSON". DSPy writes that scaffolding itself
from the field names and types, and it can rewrite it later when you
optimise. Your job is to name the fields well.

### The smallest possible DSPy program


In [ ]:
qa = dspy.Predict("question -> answer")

result = qa(question="In one sentence, what does a load balancer do?")

print("Type of result:", type(result).__name__)
print("Fields        :", list(result.keys()))
print()
print("answer:", result.answer)


### What just happened

`dspy.Predict` is a **module**: a callable object that owns one LLM call.
It took your signature, built a prompt from it, called Groq through
LiteLLM, then *parsed* the response back into named fields. You accessed
`result.answer` - not `result["choices"][0]["message"]["content"]`.

That parsing step is the quiet win. If the model returns something that
does not fit the declared output fields, DSPy is the layer that notices.

### Look at the ACTUAL prompt DSPy sent


In [ ]:
# This is the single most useful debugging call in DSPy. Do it early and often.

dspy.inspect_history(n=1)


Read that output carefully. DSPy generated:

- a **system message** describing the fields and the exact output format
  (`[[ ## answer ## ]]` markers it can parse back reliably),
- a **user message** carrying your input values.

You did not write any of it. And when you run an optimiser in notebook 03,
this is the text that changes.

### 2. Class-based Signatures

The string form is fine for toys. For real work you want the class form,
because it lets you attach a **docstring** (the task description) and
`desc=` hints per field. These are the *only* natural-language knobs you
should normally touch by hand.

### A class-based signature


In [ ]:
class Classify(dspy.Signature):
    """Classify a short customer support message."""

    message: str = dspy.InputField(desc="the raw customer message")
    category: str = dspy.OutputField(desc="one of: billing, shipping, technical, other")

classify = dspy.Predict(Classify)

for m in [
    "My card was charged twice for order 8812.",
    "The tracking number has not moved in six days.",
    "The mobile app crashes when I open settings.",
]:
    out = classify(message=m)
    print(f"{out.category:12s} <- {m}")
    time.sleep(1)   # be polite to the free tier


### 3. `Predict` vs `ChainOfThought`

`dspy.Predict` asks for the outputs directly.
`dspy.ChainOfThought` is the *same signature* with an extra `reasoning`
output field injected in front of the real ones - so the model thinks on
paper before committing.

This is the point that surprises people: switching reasoning strategy is a
**one-word code change**, not a prompt rewrite. The signature is untouched.

### Same signature, two strategies


In [ ]:
PUZZLE = ("A shop sells pens at 3 for 7 rupees. "
          "How much do 12 pens cost? Give only the number.")

direct = dspy.Predict("question -> answer")
cot = dspy.ChainOfThought("question -> answer")

a = direct(question=PUZZLE)
time.sleep(2)
b = cot(question=PUZZLE)

print("Predict        ->", a.answer.strip()[:120])
print()
print("ChainOfThought ->", b.answer.strip()[:120])
print()
print("...and CoT also exposes its reasoning field:")
print(b.reasoning.strip()[:400])


### Pitfall: CoT is not free

Chain of thought roughly doubles-to-triples output tokens. On Groq's free
tier (8000 tokens/min) that is a real budget line. Use it where the task
actually needs intermediate steps - arithmetic, multi-constraint routing,
comparisons - and skip it for lookups and classification.

You will *measure* that trade-off in notebook 03 instead of guessing.

### 4. Where DSPy sits next to the frameworks you already know

| Framework | Unit of composition | Who writes the prompt |
|---|---|---|
| LangChain | Chains of runnables | You |
| LangGraph | Graph nodes + state | You |
| CrewAI | Agents with roles/goals | You (role, goal, backstory) |
| **DSPy** | **Typed modules** | **The optimiser** |

They are not competitors so much as different layers. A common shape is:
LangGraph owns the control flow, and one node inside it is a DSPy module
whose prompt was compiled offline against a metric.

### Pitfalls recap

- **Signature field names matter.** `category` teaches the model more than
  `output`. Name fields like you would name function parameters.
- **`inspect_history()` before debugging anything.** Most "DSPy is weird"
  reports are just an unexamined generated prompt.
- **Set `temperature=0` while measuring**, or your before/after numbers are
  measuring randomness.

### Next

Notebook 02 builds the typed, multi-field signature and the metric that
notebook 03 will optimise against.